# Data Fusion

This notebook merges the **Balanced On-Chain Data** with the **Selected Off-Chain Features** to create the final training dataset for XGBoost.

## Inputs
- **On-Chain**: `labeled_days.parquet` (Address-Level, Balanced)
- **Off-Chain**: `market_daily_cleaned.parquet` (Global Daily, Top 4 Features)
- **Off-Chain**: `reddit_daily_cleaned.parquet` (Global Daily, Top 4 Features)

## Output
- `final_train_data.parquet`

## Final Feature Set

    # On-Chain (4)
    'normal_total_cnt',
    'uniq_peers_cnt',
    'burst_max_tx_5m',
    'normal_sent_cnt',

    # Reddit (3)
    'reddit_fraud_mention_ratio',
    'reddit_total_activity',
    'reddit_avg_sentiment',

    # Market (3)
    'eth_volatility_7d',
    'eth_daily_return',
    'eth_intraday_volatility'


In [9]:
import pandas as pd
import os

onchain_path = '../data/processed/chain/balanced_label_days.parquet'
market_path = '../data/processed/market/market_daily_cleaned.parquet'
reddit_path = '../data/processed/reddit/reddit_daily_cleaned.parquet'
output_dir = '../data/processed/final'
os.makedirs(output_dir, exist_ok=True)

In [10]:
chain_df = pd.read_parquet(onchain_path)
print(f"On-Chain Shape: {chain_df.shape}")

market_df = pd.read_parquet(market_path)
print(f"Off-Chain Shape: {market_df.shape}")

reddit_df = pd.read_parquet(reddit_path)
print(f"Off-Chain Shape: {reddit_df.shape}")

chain_df['day'] = pd.to_datetime(chain_df['day']).dt.date
market_df['day'] = pd.to_datetime(market_df['day']).dt.date
reddit_df['day'] = pd.to_datetime(reddit_df['day']).dt.date

On-Chain Shape: (59575, 14)
Off-Chain Shape: (1766, 4)
Off-Chain Shape: (1766, 4)


In [11]:
print("Market date range:")
print(market_df["day"].min(), "→", market_df["day"].max())
print("Duplicate days:", market_df["day"].duplicated().sum())

print("Social date range:")
print(reddit_df["day"].min(), "→", reddit_df["day"].max())
print("Duplicate days:", reddit_df["day"].duplicated().sum())

Market date range:
2017-01-01 → 2021-11-01
Duplicate days: 0
Social date range:
2017-01-01 → 2021-11-01
Duplicate days: 0


In [12]:
full_days = pd.date_range(start=market_df["day"].min(),end=market_df["day"].max(),freq="D").date

missing_market_days = set(full_days) - set(market_df["day"])
print("Missing market days:", len(missing_market_days))

full_days = pd.date_range(start=reddit_df["day"].min(),end=reddit_df["day"].max(),freq="D").date

missing_social_days = set(full_days) - set(reddit_df["day"])
print("Missing social days:", len(missing_social_days))


Missing market days: 0
Missing social days: 0


In [13]:
offchain_df = pd.merge(market_df, reddit_df, on='day', how='outer')

offchain_df = offchain_df.sort_values('day')

offchain_output_para = os.path.join(output_dir, 'offchain_daily.parquet')
offchain_df.to_parquet(offchain_output_para, index=False)

print(f"Saved off-chain dataset")
print(f"Off-chain Shape: {offchain_df.shape}")

Saved off-chain dataset
Off-chain Shape: (1766, 7)


In [14]:
merged_df = pd.merge(chain_df, offchain_df, on='day', how='left')
print(f"Full merged shape: {merged_df.shape}")

Full merged shape: (59575, 20)


In [15]:
final_features = [
    # Identity & Target
    'address', 'day', 'is_anomalous',

    # On-Chain (4)
    'normal_total_cnt',
    'uniq_peers_cnt',
    'burst_max_tx_5m',
    'normal_sent_cnt',
    # Reddit (4)
    'reddit_fraud_mention_ratio',
    'reddit_total_activity',
    'reddit_avg_sentiment',
    # Market (4)
    'eth_volatility_7d',
    'eth_daily_return',
    'eth_intraday_volatility'
]

train_df = merged_df[final_features].copy()

print(f"Final Training Set Shape: {train_df.shape}")
print("\nClass Distribution:")
print(train_df['is_anomalous'].value_counts())

train_df.head()

Final Training Set Shape: (59575, 13)

Class Distribution:
is_anomalous
0    29904
1    29671
Name: count, dtype: int64


,address,day,is_anomalous,normal_total_cnt,uniq_peers_cnt,burst_max_tx_5m,normal_sent_cnt,reddit_fraud_mention_ratio,reddit_total_activity,reddit_avg_sentiment,eth_volatility_7d,eth_daily_return,eth_intraday_volatility
0,0xd624d046edbdef805c5e4140dce5fb5ec1b39a3c,2017-03-15,0,2,2,2,1,0.077586,580,0.371972,0.080752,0.223395,0.231198
1,0xc859d7753a0295d8b88be34014a4956e15a2b745,2017-03-28,0,2,2,2,1,0.069388,735,0.338754,0.086855,0.022309,0.036435
2,0xc859d7753a0295d8b88be34014a4956e15a2b745,2017-03-29,0,1,1,2,1,0.055385,650,0.298511,0.084683,0.046306,0.065662
3,0xbad41e5412ebaf22b8ec127521060fd9fa29bbdd,2017-04-14,0,2,2,2,1,0.073171,369,0.324839,0.050267,-0.052675,0.065790
4,0xcda94feb23fbf1c4ca73b45de92e1a4cc2bd40ab,2017-05-16,0,1,1,2,0,0.064205,623,0.304364,0.020178,-0.032212,0.048031


In [16]:
output_file_para = os.path.join(output_dir, 'final_train_data.parquet')
train_df.to_parquet(output_file_para, index=False)